# Train Models on MIMIC-IV

In [ ]:
from pathlib import Path

import numpy as np
import pytorch_lightning as pl
import yaml

In [ ]:
from lab_ecg import classification, config, dataset

### Data

In [ ]:
# select an arbitrary parameter/threshold combination
param_threshold = ("albumin", "<3.4")

In [ ]:
store = dataset.HDFSignalDatastore(
    "data/mimic4_samples.h5",
    "data/mimic4_signals.h5",
    src_channels=["I", "II", "III", "aVR", "aVF", "aVL", "V1", "V2", "V3", "V4", "V5", "V6"],
    # channels=["II"],  # uncomment to train single-lead model; you can actually pass an arbitrary list of channels
    label_cols=[param_threshold[0]],
    binary_thresholds=[param_threshold],
    fs=500,  # sample rate of source signals => do not change!
    memory_limit="40GB",
)
store.feature_cols = ["subject_id", "sample_id", "male", "age"]
len(store)

### Config

In [ ]:
# get the default config used in the paper
# instead of "xresnet1d101", you can also pass "kwon"
params = config.get_default_config("xresnet1d101", "checkpoints")

In [ ]:
transform, test_transform = config.build_pipeline(params, store)

### Data Loaders

In [ ]:
split = params["data_store"]["label_cols"][0] + "_split"
train_indices = np.flatnonzero((store.df[split] == "train") | store.df[split].str.startswith("train_").fillna(False))
val_indices = np.flatnonzero(store.df[split].isin(["val"]))
(len(train_indices), len(val_indices))

In [ ]:
store.balance(indices=train_indices, add_to_feature_cols=True)
store.df["sample_weight"].value_counts(dropna=False)

In [ ]:
train_loader = dataset.SignalDataset(store, indices=train_indices, transform=transform, auto_open=True).dataloader(
    batch_size=params["training"]["batch_size"],
    sampler="shuffle",
    num_workers=0 if store.in_memory else 10,
    drop_last="if_1",
)
len(train_loader.dataset)

In [ ]:
val_loader = dataset.SignalDataset(store, indices=val_indices, transform=test_transform, auto_open=True).dataloader(
    batch_size=params["training"]["batch_size"], num_workers=0 if store.in_memory else 4
)
len(val_loader.dataset)

### Training

In [ ]:
Path(params["paths"]["checkpoints"]).mkdir(exist_ok=True, parents=True)

In [ ]:
target_metric = params["training"].get("target_metric", "val_loss")
metric_mode = "min" if any(s in target_metric for s in ("_loss", "_error", "_mse", "_rmse", "_mae")) else "max"

In [ ]:
trainer = pl.Trainer(
    max_epochs=params["training"]["epochs"],
    logger=pl.loggers.CSVLogger(params["paths"]["checkpoints"]),
    log_every_n_steps=50,
    gradient_clip_val=params["training"].get("gradient_clip", 1),
    accelerator="gpu",
    devices=1,
    callbacks=[
        pl.callbacks.ModelCheckpoint(
            monitor=target_metric,
            mode=metric_mode,
            save_last=True,
            dirpath=params["paths"]["checkpoints"],
            filename="{epoch}-{step}-{val_loss:.2f}",
        ),
        pl.callbacks.EarlyStopping(
            monitor=target_metric, mode=metric_mode, patience=params["training"].get("patience", 0), strict=False
        ),
    ],
)

In [ ]:
with open(params["paths"]["checkpoints"] + "/params.yaml", mode="wt") as stream:
    yaml.dump(params, stream)

In [ ]:
model = classification.make(params, train_dataset=train_loader.dataset)

In [ ]:
# all parameters
sum(np.prod(p.shape) for p in model.network.parameters())

In [ ]:
trainer.fit(model, train_loader, val_loader)

In [ ]:
train_loader.dataset.close()
val_loader.dataset.close();

For evaluating the trained model on held-out test data, either from MIMIC-IV or MIMIC-III, see `evaluate.ipynb`.